# Decorte Baselines

Install all necessary packages to run and analyze this model

In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install matplotlib datasets pandas tqdm sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


Check if we run on CUDA

In [3]:
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.rand(3, 3).to(device)
print(x.device)  # Should output 'cuda:0'

True
cuda:0


Let's do a set up

In [4]:
from pathlib import Path

DATA_PATH = Path("./data/")

Load the dataset.

In [5]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Load pre-trained sentence transformer model
EMBEDDING_MODEL = "ElenaSenger/career-path-representation-mpnet-decorte"
model = SentenceTransformer(EMBEDDING_MODEL)

# Data
### Load data for neural transformation training
print("Loading data...")

# Load the dataset
dataset = load_dataset("jensjorisdecorte/anonymous-working-histories")

/home/nikita/projects/CareerGuide/experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3520.30it/s]


Loading data...


Replace some ESCO titles.

In [6]:
import pandas as pd

def replace_esco_titles(example, i):
    """
    Replaces specific ESCO job titles with alternative titles for consistency.

    Args:
        example (dict): A dictionary representing a dataset row.
        i (int): The index of the ESCO title column.

    Returns:
        dict: Updated dictionary with the replaced ESCO title and URI.
    """
    replacements_title = {
        'ICT security engineer': 'cyber incident responder',
        'ict security engineer': 'cyber incident responder',
        'care at home worker': 'care home worker',
        'residential care home worker': 'care home worker',
        'ICT security manager': 'cybersecurity risk manager',
        'ict security manager': 'cybersecurity risk manager',
        'care at hmoe worker': 'care home worker',
        'handyman': 'handyperson',
        'corporate banking manager': 'corporate banking adviser',
    }

    original_title = example[f'ESCO_title_{i}']
    if not pd.isna(original_title):
        processed_title = original_title.strip().lower()
        final_title = replacements_title.get(processed_title, processed_title)
    else:
        final_title = original_title

    example[f'ESCO_title_{i}'] = final_title

    replacements_uri = {
        'http://data.europa.eu/esco/occupation/81309031-dad2-4a7a-bde6-7f6e518f89ff': 
        'http://data.europa.eu/esco/occupation/f4525ed8-54eb-4a3b-90db-55cc01b0d9fd'
    }
    
    example[f'ESCO_uri_{i}'] = replacements_uri.get(example[f'ESCO_uri_{i}'], example[f'ESCO_uri_{i}'])
    
    return example

# Apply replacements to all columns in the dataset beginning with ESCO_title
for i in range(16):
    dataset['train'] = dataset['train'].map(lambda example: replace_esco_titles(example, i))
    dataset['validation'] = dataset['validation'].map(lambda example: replace_esco_titles(example, i))
    dataset['test'] = dataset['test'].map(lambda example: replace_esco_titles(example, i))


Create ESCO occupations dictionary.

In [7]:
# Load descriptions for ESCO occupations
ESCO_occupations = pd.read_csv(DATA_PATH / "occupations_en.csv")


# Create dictionary for ESCO occupations
ESCO_occupations_dict = ESCO_occupations.set_index("conceptUri")[
    "description"
].to_dict()

# Add to ESCO_occupations_dict keys which are the names of the occupations, and as value the description of the occupation
ESCO_occupations_dict.update(
    ESCO_occupations.set_index("preferredLabel")["description"].to_dict()
)

# For every occupation, go through the altLabels and add them to the dictionary
for index, row in ESCO_occupations.iterrows():
    # If there are no altLabels, skip
    if pd.isna(row["altLabels"]):
        continue
    for alt_label in row["altLabels"].split("\n"):
        ESCO_occupations_dict[alt_label] = row["description"]

Find the latest start/end dates across all experiences per split, then collect each person's last experience to inspect the most recent ones per split.

In [8]:
def latest_date(split, field):                  
      best_key = None                             
      best_str = None                         
      for person in split:                        
          for i in range(person["number_of_experiences"]):         
              v = person[f"{field}_{i}"]  
              if v is None or v == "current":     
                  continue                        
              m, y = map(int, v.split("/"))
              key = (y, m)                        
              if best_key is None or key > best_key:
                  best_key = key                  
                  best_str = v
      return best_str                             
                  
                                              
for split_name, split in dataset.items():
      print(f"Split: {split_name}")
      print(f"Latest start: {latest_date(split,   
  'start')}")
      print(f"Latest end:   {latest_date(split,   
  'end')}")                                   
      print()                                  


def parse_date(s):                              
      if s is None or s == "current":           
          return None                         
      m, y = map(int, s.split("/"))       
      return (y, m)
                                                  
                                          
for split_name, split in dataset.items():       
      latest_experiences = []                     
      for person in split:                        
          last = person["number_of_experiences"] -   1                                              
          start = person[f"start_{last}"]         
          key = parse_date(start)             
          if key is None:                         
              continue
          latest_experiences.append((             
              key,
              person["identifier"],               
              start,                      
              person[f"end_{last}"],
          ))                                      
   
      latest_experiences.sort(key=lambda r: r[0]) 
                                          
      print(f"Split: {split_name}")
      for _, identifier, start, end in latest_experiences[:10]:
          print(f"  {identifier}  start={start}     end={end}")                                 
      print()  



Split: train
Latest start: 05/2021
Latest end:   08/2021

Split: validation
Latest start: 12/2020
Latest end:   05/2021

Split: test
Latest start: 10/2020
Latest end:   02/2021

Split: train
  94417768  start=04/1984     end=current
  14585273  start=01/1994     end=01/2008
  21629057  start=05/1994     end=05/2000
  30083943  start=07/1994     end=08/2015
  13411858  start=02/1995     end=current
  27689009  start=06/1995     end=current
  30083884  start=01/1996     end=current
  28243590  start=12/1996     end=current
  19147603  start=01/1997     end=04/2014
  30127072  start=01/1997     end=01/2002

Split: validation
  28398216  start=01/1997     end=04/2014
  24709432  start=04/2000     end=current
  26098594  start=01/2001     end=current
  26975573  start=01/2001     end=02/2011
  11813872  start=01/2003     end=current
  24592627  start=03/2004     end=09/2014
  33803142  start=01/2005     end=01/2015
  15553584  start=02/2005     end=05/2005
  36149549  start=11/2005     end=

Find earliest start and latest end.

In [9]:
def extreme_date_all_splits(dataset, field, mode):
    best_key = None
    best_str = None
    for split in dataset.values():
        for person in split:
            for i in range(person["number_of_experiences"]):
                v = person[f"{field}_{i}"]
                if v is None or v == "current":
                    continue
                m, y = map(int, v.split("/"))
                key = (y, m)
                if best_key is None or (key < best_key if mode == "earliest" else key > best_key):
                    best_key = key
                    best_str = v
    return best_str


print(f"Earliest start (all splits): {extreme_date_all_splits(dataset, 'start', 'earliest')}")
print(f"Latest end     (all splits): {extreme_date_all_splits(dataset, 'end', 'latest')}")


Earliest start (all splits): 02/1753
Latest end     (all splits): 08/2021


Inspect start dates earlier than 1950.

In [10]:
for split_name, split in dataset.items():
    for person in split:
        for i in range(person["number_of_experiences"]):
            v = person[f"start_{i}"]
            if v is None or v == "current":
                continue
            _, y = map(int, v.split("/"))
            if y < 1950:
                print(f"{split_name}  person_id={person['identifier']}  start={v}")


train  person_id=61677751  start=02/1753
train  person_id=61677751  start=02/1753
train  person_id=61677751  start=02/1753


Records only found for person with id 61677751. Remove it.

In [11]:
dataset = dataset.filter(lambda row: row["identifier"] != 61677751)


Filter out people with less than 2 experiences.  
Explode each remaining person's wide-format experiences into one row per experience.

In [12]:
from datasets import Dataset, DatasetDict

def explode_experiences(split):
    rows = []
    for person in split:
        for i in range(person["number_of_experiences"]):
            rows.append({
                "experience_id": person[f"uuid_{i}"],
                "person_id": person["identifier"],
                "experience_number": i,
                "industry": person["industry"],
                "title": person[f"title_{i}"],
                "description": person[f"description_{i}"],
                "ESCO_uri": person[f"ESCO_uri_{i}"],
                "ESCO_title": person[f"ESCO_title_{i}"].strip(),
                "start": person[f"start_{i}"],
                "end": person[f"end_{i}"]
            })
    return Dataset.from_list(rows)

dataset = dataset.filter(lambda row:            
  row["number_of_experiences"] >= 2)
dataset = DatasetDict({
    split: explode_experiences(dataset[split]) for split in dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end'],
        num_rows: 7908
    })
    validation: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end'],
        num_rows: 957
    })
    test: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end'],
        num_rows: 1050
    })
})


Replace "current" end dates with start+1 year (capped at 8/2021), then compute a months_of_experience field as the month difference between start and end for every experience row.

In [13]:
DATE_CAP = (2021, 8)

def months_between(start, end):
    sm, sy = map(int, start.split("/"))
    em, ey = map(int, end.split("/"))
    return (ey - sy) * 12 + (em - sm)

def fill_current_end(row):              
    if row["end"] != "current":                 
        return row                              
    m, y = map(int, row["start"].split("/"))    
    new_y, new_m = y + 1, m                     
    if (new_y, new_m) > DATE_CAP:                    
        new_y, new_m = DATE_CAP                      
    row["end"] = f"{new_m}/{new_y}"             
    return row
                                                  
                                              
dataset = dataset.map(fill_current_end)
dataset = dataset.map(lambda row: {"months_of_experience": months_between(row["start"], row["end"])})

Map: 100%|██████████| 1050/1050 [00:00<00:00, 23659.59 examples/s]


Inspect people with less than 0 months of experience.

In [14]:
for split_name, split in dataset.items():
    bad = [r for r in split if r["months_of_experience"] <= 0]
    print(f"{split_name}: {len(bad)} experiences with months_of_experience <= 0")
    for r in bad[:20]:
        print(f"  person_id={r['person_id']}  exp#={r['experience_number']}  "
              f"start={r['start']}  end={r['end']}  months={r['months_of_experience']}")


train: 83 experiences with months_of_experience <= 0
  person_id=47729453  exp#=1  start=01/2006  end=01/2006  months=0
  person_id=47729453  exp#=3  start=01/2007  end=01/2007  months=0
  person_id=47729453  exp#=5  start=01/2009  end=01/2009  months=0
  person_id=47729453  exp#=9  start=01/2016  end=01/2016  months=0
  person_id=18488289  exp#=1  start=06/2009  end=06/2009  months=0
  person_id=91318828  exp#=2  start=08/2014  end=01/2014  months=-7
  person_id=23497307  exp#=2  start=07/2017  end=03/2017  months=-4
  person_id=61319162  exp#=2  start=01/2008  end=01/2008  months=0
  person_id=22754014  exp#=0  start=01/2006  end=01/2006  months=0
  person_id=30642458  exp#=4  start=10/2013  end=10/2013  months=0
  person_id=45462344  exp#=7  start=08/2014  end=01/2011  months=-43
  person_id=10235429  exp#=6  start=01/2008  end=01/2008  months=0
  person_id=11522068  exp#=0  start=06/2011  end=06/2011  months=0
  person_id=11522068  exp#=1  start=06/2011  end=06/2011  months=0
  per

Remove people with less than 0 months of experience.

In [15]:
bad_persons = {
    split_name: {r["person_id"] for r in split if r["months_of_experience"] < 0}
    for split_name, split in dataset.items()
}
for split_name, ids in bad_persons.items():
    print(f"{split_name}: dropping {len(ids)} persons with negative-month experiences")

dataset = DatasetDict({
    split_name: split.filter(lambda r: r["person_id"] not in bad_persons[split_name])
    for split_name, split in dataset.items()
})
print(dataset)


train: dropping 17 persons with negative-month experiences
validation: dropping 5 persons with negative-month experiences
test: dropping 3 persons with negative-month experiences


Filter: 100%|██████████| 1050/1050 [00:00<00:00, 101505.50 examples/s]

DatasetDict({
    train: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end', 'months_of_experience'],
        num_rows: 7815
    })
    validation: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end', 'months_of_experience'],
        num_rows: 926
    })
    test: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end', 'months_of_experience'],
        num_rows: 1039
    })
})


Transform free text experiences (history) and ESCO experience (target).

In [16]:
def free_text_experience(_experience_title, _experience_description):
    return f"role: {_experience_title} \n description: {_experience_description}"

def ESCO_experience(_ESCO_title, _ESCO_uri):
    try:
        return f"esco role: {_ESCO_title} \n description: {ESCO_occupations_dict[_ESCO_uri]}"
    except KeyError:
        return f"esco role: {_ESCO_title} \n description: {ESCO_occupations_dict[_ESCO_title]}"
    
dataset = dataset.map(lambda row: {             
      "free_text_experience":                     
  free_text_experience(row["title"],          
  row["description"]),                            
      "ESCO_experience":
  ESCO_experience(row["ESCO_title"],              
  row["ESCO_uri"]),                       
  })

Map: 100%|██████████| 1039/1039 [00:00<00:00, 14275.59 examples/s]


Group experiences by person, sort them chronologically, then for each person enumerate every contiguous subspan of length ≥2 as a (prefix history, prefix months, industry, target ESCO) tuple — keeping only the last 16 per person — and build a shared industry vocabulary across all splits.

In [17]:
from collections import defaultdict


def build_pairs(split):
    by_person = defaultdict(list)
    for row in split:
        by_person[row["person_id"]].append(row)

    pairs = []
    for exps in by_person.values():
        exps.sort(key=lambda r: r["experience_number"])
        n = len(exps)
        industry = exps[0]["industry"] if exps[0]["industry"] is not None else "<unk>"
        person_pairs = []
        for L in range(2, n + 1):           # subspan length
            for j in range(n - L + 1):      # start position
                prefix = [r["free_text_experience"] for r in exps[j : j + L - 1]]
                prefix_months = [r["months_of_experience"]  for r in exps[j : j + L - 1]]
                target = exps[j + L - 1]["ESCO_experience"]
                person_pairs.append((prefix, prefix_months, industry, target))
        pairs.extend(person_pairs[-16:])

    return pairs


pairs = {split: build_pairs(dataset[split]) for split in dataset}

# Industry vocabulary: built once over all splits so val/test industries are known.
_industries_sorted = sorted({p[2] for split_pairs in pairs.values() for p in split_pairs})
industry_to_id = {ind: i for i, ind in enumerate(_industries_sorted)}
n_industries = len(industry_to_id)
print(f"industries: {n_industries}")

for split, p in pairs.items():
    print(f"{split}: {len(p)} pairs")


industries: 24
train: 13311 pairs
validation: 1489 pairs
test: 1787 pairs


Encode every prefix experience text and target ESCO text into embeddings, build a log1p(months) duration scalar per step, and map each pair's industry to an integer id — returning per-pair history embeddings, month tensors, industry ids, and ESCO target embeddings for train and validation.

In [18]:
import torch

def encode_pairs(pairs):
    prefix_texts_list, prefix_months_list, industries_list, esco_texts = zip(*pairs)

    print("Embedding career history and ESCO occupation texts...")

    # Encode each experience text once — one step per role (no month-duplicate inflation).
    lengths = [len(seq) for seq in prefix_texts_list]
    flat_texts = [t for seq in prefix_texts_list for t in seq]
    flat_embeddings = model.encode(
        flat_texts,
        batch_size=256,
        convert_to_tensor=True,
        show_progress_bar=True,
    )
    per_pair_embeddings = list(torch.split(flat_embeddings, lengths))

    # Per-step duration feature — log1p(months) as a float scalar on the same device.
    per_pair_months = [
        torch.log1p(torch.tensor(m, dtype=torch.float32, device=flat_embeddings.device))
        for m in prefix_months_list
    ]

    # Per-pair industry id (long scalar). Unknown industries map to the '<unk>' bucket.
    industry_ids = torch.tensor(
        [industry_to_id.get(ind, industry_to_id.get("<unk>", 0)) for ind in industries_list],
        dtype=torch.long,
    )

    esco_occupation_embeddings = model.encode(
        list(esco_texts),
        batch_size=256,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    return per_pair_embeddings, per_pair_months, industry_ids, esco_occupation_embeddings


val_history_embeddings, val_history_months, val_industry_ids, val_esco_embeddings = encode_pairs(pairs["validation"])

embedding_dim = val_esco_embeddings.shape[1]
val_lens = [t.size(0) for t in val_history_embeddings]
print(f"Val: {len(val_history_embeddings)} sequences, embedding_dim={embedding_dim}")
print(f"  prefix lens - min={min(val_lens)}  max={max(val_lens)}  mean={sum(val_lens)/len(val_lens):.1f}")


Embedding career history and ESCO occupation texts...


Batches: 100%|██████████| 6/6 [00:01<00:00,  3.30it/s]

Val: 1489 sequences, embedding_dim=768
  prefix lens - min=1  max=14  mean=2.6


Wrap the per-pair history embeddings, month scalars, industry ids, and ESCO targets into a CareerHistoryDataset, and build train/val DataLoaders with a collate_pad function that pads variable-length history sequences (and their months) to a common batch length while stacking industry ids, true lengths, and targets.

In [19]:
import torch.nn.functional as F

_all_esco_texts = sorted({target for sp in pairs.values() for *_, target in sp})
_text_to_label_id = {t: i for i, t in enumerate(_all_esco_texts)}
_label_bank = model.encode(
    _all_esco_texts, batch_size=256, convert_to_tensor=True, show_progress_bar=False
)
_label_bank = F.normalize(_label_bank, dim=-1).to(device)

In [20]:
# 1. Encode test pairs (same pipeline as train/val).
test_history_embeddings, test_history_months, test_industry_ids, test_esco_embeddings = encode_pairs(pairs["test"])

Embedding career history and ESCO occupation texts...


Batches: 100%|██████████| 7/7 [00:02<00:00,  3.44it/s]


No training, no model — just retrieve against the ESCO label bank using:
- **last-experience**: the final prefix embedding (what's the value of the RNN if this already works?)
- **mean-pool**: unweighted mean of the month-exploded prefix embeddings (= tenure-weighted mean of original embeddings)

In [21]:
import torch

@torch.no_grad()
def baseline_metrics(history_embs, pair_list, pool):
    """
    pool: "last" = final prefix embedding, "mean" = mean of month-exploded prefix.
    Retrieves against _label_bank (all splits' ESCO labels, L2-normalized).
    """
    if pool == "last":
        preds = torch.stack([h[-1] for h in history_embs]).to(device)
    elif pool == "mean":
        preds = torch.stack([h.mean(dim=0) for h in history_embs]).to(device)
    else:
        raise ValueError(pool)
    preds = F.normalize(preds, dim=-1)

    true_ids = torch.tensor(
        [_text_to_label_id[t] for *_, t in pair_list], device=device
    )
    scores = preds @ _label_bank.T
    true_scores = scores.gather(1, true_ids.unsqueeze(1))
    ranks = (scores > true_scores).sum(dim=1) + 1

    mrr_val = (1.0 / ranks.float()).mean().item()
    r5  = (ranks <= 5).float().mean().item()
    r10 = (ranks <= 10).float().mean().item()
    return {"MRR": round(mrr_val, 4), "R@5": round(r5, 4), "R@10": round(r10, 4)}


print("=== Validation ===")
print(f"last-exp : {baseline_metrics(val_history_embeddings,  pairs['validation'], 'last')}")
print(f"mean-pool: {baseline_metrics(val_history_embeddings,  pairs['validation'], 'mean')}")

print("\n=== Test ===")
print(f"last-exp : {baseline_metrics(test_history_embeddings, pairs['test'],       'last')}")
print(f"mean-pool: {baseline_metrics(test_history_embeddings, pairs['test'],       'mean')}")


=== Validation ===
last-exp : {'MRR': 0.2303, 'R@5': 0.3002, 'R@10': 0.3902}
mean-pool: {'MRR': 0.2545, 'R@5': 0.3398, 'R@10': 0.4385}

=== Test ===
last-exp : {'MRR': 0.2354, 'R@5': 0.3408, 'R@10': 0.4208}
mean-pool: {'MRR': 0.2575, 'R@5': 0.3598, 'R@10': 0.4522}


No-cosine, no-model baseline: rank ESCO URIs from the prefix **by recency** (last role → 2nd-last → … , deduplicated). Top-1 is the strict "ESCO of last workplace" prediction; deeper ranks make MRR / R@5 / R@10 meaningful. Targets that never appear in the prefix get rank = ∞ (contribute 0).

In [22]:
def last_role_uri_ranks(split):
    """For every (prefix, target) pair, return target's rank in the prefix's
    URIs ordered by recency (most recent first, deduplicated). None if absent."""
    by_person = defaultdict(list)
    for row in split:
        by_person[row["person_id"]].append(row)

    all_ranks = []
    for exps in by_person.values():
        exps.sort(key=lambda r: r["experience_number"])
        n = len(exps)
        person_ranks = []
        for L in range(2, n + 1):
            for j in range(n - L + 1):
                prefix_uris = [exps[k]["ESCO_uri"] for k in range(j + L - 2, j - 1, -1)]
                target_uri = exps[j + L - 1]["ESCO_uri"]

                seen, ranked = set(), []
                for u in prefix_uris:
                    if u not in seen:
                        seen.add(u)
                        ranked.append(u)

                rank = ranked.index(target_uri) + 1 if target_uri in seen else None
                person_ranks.append(rank)
        all_ranks.extend(person_ranks[-16:])
    return all_ranks


def last_role_uri_metrics(split_name):
    ranks = last_role_uri_ranks(dataset[split_name])
    n = len(ranks)
    mrr = sum(1.0 / r for r in ranks if r is not None) / n
    r1  = sum(1 for r in ranks if r is not None and r <= 1)  / n
    r5  = sum(1 for r in ranks if r is not None and r <= 5)  / n
    r10 = sum(1 for r in ranks if r is not None and r <= 10) / n
    return {"MRR": round(mrr, 4), "R@1": round(r1, 4), "R@5": round(r5, 4), "R@10": round(r10, 4)}


print("=== Validation ===")
print(f"last-role-URI (recency-ranked): {last_role_uri_metrics('validation')}")

print("\n=== Test ===")
print(f"last-role-URI (recency-ranked): {last_role_uri_metrics('test')}")

=== Validation ===
last-role-URI (recency-ranked): {'MRR': 0.2256, 'R@1': 0.1827, 'R@5': 0.2794, 'R@10': 0.2814}

=== Test ===
last-role-URI (recency-ranked): {'MRR': 0.2066, 'R@1': 0.1623, 'R@5': 0.2619, 'R@10': 0.263}
